# Notebook 03: In-Sample Validation

This notebook validates the continuous Hidden Markov Model (CHMM, no jumps) by checking
whether simulated paths reproduce the stylized facts of the observed return series.
The number of hidden states K and the number of Monte Carlo paths N_PATHS are tunable
so the user can interactively explore their effect on fit quality.

### Validation Metrics
1. **KS pass rate** (%) — Kolmogorov-Smirnov test at alpha = 0.05
2. **Excess kurtosis** — simulated vs observed
3. **ACF-MAE** — mean absolute error of ACF(|G_t|) over lags 1-252
4. **Wasserstein-1 distance** — mean earth-mover distance
5. **Hellinger distance** — histogram overlap distance

### Outputs
- **Figure 3**: Three-panel IS comparison (density, ACF, Q-Q)
- **Trajectory example**: Observed vs one simulated path
- **ACF comparison**: Raw and absolute ACF overlay via `plot_acf_comparison`

## Setup

In [ ]:
include("../Include.jl");

## Configuration (Tunable)

In [ ]:
# --- TUNABLE PARAMETERS ---
ticker = "SPY";
K = 6;                  # number of hidden states
MAX_ITER = 60;          # Baum-Welch iterations
N_PATHS = 1000;         # Monte Carlo paths
L = 252;                # ACF max lag (1 trading year)
risk_free_rate = 0.0;
Δt = 1/252;

## Load Data and Train CHMM

In [ ]:
# --- LOAD DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_excess_return_matrix = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
Rᵢ = findfirst(x -> x == ticker, list_of_all_tickers) |> i -> all_firms_excess_return_matrix[:, i];
in_sample_data = Rᵢ[1:(maximum_number_trading_days-1)];

println("Training data: $(length(in_sample_data)) observations for $ticker")

In [ ]:
# --- TRAIN CHMM (no jumps) ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = in_sample_data,
    number_of_states = K,
    max_iter = MAX_ITER
));

println("Training complete: $(length(model.log_likelihood_history)) EM iterations, K=$K")

In [ ]:
# --- STATIONARY DISTRIBUTION ---
T_matrix = zeros(K, K);
for i in 1:K
    T_matrix[i, :] = probs(model.transition[i]);
end

π_stationary = (T_matrix^1000)[1, :];
start_dist = Categorical(π_stationary);
n_steps = length(in_sample_data);

println("Stationary distribution computed. Sum = $(round(sum(π_stationary), digits=6))")

## Simulate N_PATHS Paths

In [ ]:
# --- SIMULATE ---
decoded_is = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(start_dist);
    states = model(s0, n_steps);
    for j in 1:n_steps
        decoded_is[j, i] = rand(model.emission[states[j]]);
    end
end

println("CHMM: $(N_PATHS) paths of length $(n_steps) simulated.")

## Validation Metrics

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function compute_is_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                            α=0.05, L=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    acf_obs = autocor(abs.(observed), 1:L);

    # Per-path accumulators
    ks_pass = 0;
    kurt_sum = 0.0; acf_mae_sum = 0.0;
    w1_sum = 0.0; hell_sum = 0.0;

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end

        # Excess kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        # ACF-MAE on absolute returns
        acf_sim = autocor(abs.(sim), 1:L);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));

        # Hellinger distance (histogram-based)
        edges = range(minimum([observed; sim]) - 10, maximum([observed; sim]) + 10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end

    # Build results
    ks_rate = round(100 * ks_pass / n_paths, digits=1);
    kurt_sim = round(kurt_sum / n_paths, digits=2);
    kurt_o = round(kurt_obs, digits=2);
    acf_mae = round(acf_mae_sum / n_paths, digits=4);
    w1 = round(w1_sum / n_paths, digits=3);
    hell = round(hell_sum / n_paths, digits=4);

    return (ks_rate=ks_rate, kurtosis_sim=kurt_sim, kurtosis_obs=kurt_o,
            acf_mae=acf_mae, wasserstein=w1, hellinger=hell)
end;

In [ ]:
# --- COMPUTE AND PRINT METRICS ---
metrics = compute_is_metrics(in_sample_data, decoded_is; L=L);

println("╔══════════════════════════════════════════════════════╗");
println("║  In-Sample Validation: CHMM (K=$K, $ticker)          ║");
println("╠══════════════════════════════════════════════════════╣");
println("║  KS pass rate (α=0.05):  $(lpad(metrics.ks_rate, 8))%              ║");
println("║  Excess kurtosis (sim):  $(lpad(metrics.kurtosis_sim, 8))               ║");
println("║  Excess kurtosis (obs):  $(lpad(metrics.kurtosis_obs, 8))               ║");
println("║  ACF-MAE |Gₜ|:          $(lpad(metrics.acf_mae, 8))               ║");
println("║  Wasserstein-1:          $(lpad(metrics.wasserstein, 8))               ║");
println("║  Hellinger:              $(lpad(metrics.hellinger, 8))               ║");
println("╚══════════════════════════════════════════════════════╝");

## Figure 3: In-Sample Comparison

Three-panel figure:
- **(a)** Density: histogram of observed + density of simulated
- **(b)** ACF(|G_t|): observed vs simulated mean with 10-90th percentile band
- **(c)** Q-Q plot: observed vs simulated quantiles (0.1st to 99.9th)

In [ ]:
# --- FIGURE 3a: MARGINAL DENSITY ---
p3a = plot(title="(a) Marginal Density (IS KS: $(metrics.ks_rate)%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");

histogram!(p3a, in_sample_data, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");
density!(p3a, vec(decoded_is), lw=2, color=:blue, alpha=0.7, label="CHMM (K=$K)");
xlims!(p3a, -800, 800);

# --- FIGURE 3b: ACF(|G|) COMPARISON ---
τ = 1:L;
acf_obs = autocor(abs.(in_sample_data), τ);

n_acf_paths = min(200, N_PATHS);
acf_archive = hcat([autocor(abs.(decoded_is[:, i]), τ) for i in 1:n_acf_paths]...);
acf_mean = mean(acf_archive, dims=2)[:];
acf_p10 = [quantile(acf_archive[t, :], 0.10) for t in 1:L];
acf_p90 = [quantile(acf_archive[t, :], 0.90) for t in 1:L];

p3b = plot(τ, acf_obs, lw=2, color=:red, ls=:dash, label="Observed",
    title="(b) ACF(|Gₜ|) — Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p3b, τ, acf_mean, lw=2, color=:blue, label="CHMM mean");
plot!(p3b, τ, acf_p10, fillrange=acf_p90, alpha=0.15, color=:blue, label="10-90th pctl");

# --- FIGURE 3c: TAIL Q-Q PLOT ---
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(in_sample_data, probs_qq);
q_sim = quantile(vec(decoded_is), probs_qq);

p3c = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Tail Q-Q Plot (0.1st-99.9th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p3c, q_obs, q_sim, ms=3, alpha=0.6, color=:blue, label="CHMM (K=$K)");

# --- COMBINE ---
fig3 = plot(p3a, p3b, p3c, layout=(1, 3), size=(1400, 400),
    plot_title="Figure 3: In-Sample Comparison — $(ticker) (K=$K)",
    plot_titlefontsize=12);
display(fig3);

savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-IS-CHMM-$(ticker)-K$(K).svg"));
savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-IS-CHMM-$(ticker)-K$(K).pdf"));

## Trajectory Example

Overlay the observed return series against one randomly selected simulated path.

In [ ]:
# --- TRAJECTORY VISUALIZATION ---
idx = rand(1:N_PATHS);
n_show = min(500, n_steps);

p_traj = plot(in_sample_data[1:n_show], lw=1, color=:red, alpha=0.6, label="Observed",
    title="Return Trajectory: Observed vs Simulated (first $n_show steps, path $idx)",
    titlefontsize=10, xlabel="Trading Day", ylabel="Excess Growth Rate");
plot!(p_traj, decoded_is[1:n_show, idx], lw=1, color=:blue, alpha=0.6, label="CHMM (K=$K)");
display(p_traj);

savefig(p_traj, joinpath(_PATH_TO_FIGURES, "Fig-IS-Trajectory-$(ticker)-K$(K).svg"));

## ACF Comparison

Single path ACF overlay (raw returns and absolute returns) using `plot_acf_comparison`.

In [ ]:
# --- ACF COMPARISON: RAW RETURNS ---
sim_path = decoded_is[:, idx];

p_acf_raw = plot_acf_comparison(in_sample_data, sim_path,
    "ACF of Raw Returns — $(ticker) (K=$K)", idx; is_absolute=false, L=L);

# --- ACF COMPARISON: ABSOLUTE RETURNS ---
p_acf_abs = plot_acf_comparison(in_sample_data, sim_path,
    "ACF of |Gₜ| — $(ticker) (K=$K)", idx; is_absolute=true, L=L);

p_acf_combined = plot(p_acf_raw, p_acf_abs, layout=(1, 2), size=(1200, 400));
display(p_acf_combined);

savefig(p_acf_combined, joinpath(_PATH_TO_FIGURES, "Fig-IS-ACF-$(ticker)-K$(K).svg"));

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.